# LineVul Attention Distance Server
Loads the LineVul (CodeBERT fine-tuned) model and serves an API for computing
attention-based distances for directed fuzzing. Exposed via ngrok on **Google Colab GPU T4**.

**Endpoints:**
- `POST /score_blocks` — returns normalized attention scores `w(m)` per basic block
- `POST /compute_distances` — returns final attention distances `db_att = db_phys × (1.5 - w(m))`
- `GET /health`
- `GET /metrics`

In [ ]:
# ── 1. Install dependencies ──────────────────────────────────────────────────
!pip install -q fastapi uvicorn pyngrok transformers torch gdown

In [ ]:
# ── 2. Authenticate ngrok ────────────────────────────────────────────────────
# Store NGROK_TOKEN in Colab: left sidebar → key icon → "Secrets"
from google.colab import userdata
from pyngrok import ngrok, conf

NGROK_TOKEN = userdata.get("NGROK_TOKEN")
conf.get_default().auth_token = NGROK_TOKEN

In [ ]:
# ── 3. Load LineVul model ─────────────────────────────────────────────────────
# LineVul is NOT on HuggingFace. Weights are distributed by the authors on
# Google Drive. The base architecture is microsoft/codebert-base (RoBERTa),
# and the fine-tuned head is loaded on top via load_state_dict.
import os
import argparse
import torch
import torch.nn as nn
from torch.nn import CrossEntropyLoss
from transformers import (
    RobertaConfig,
    RobertaTokenizer,
    RobertaForSequenceClassification,
 )

# ── Inline linevul_model.py (awsm-research/LineVul) ──────────────────────────
class RobertaClassificationHead(nn.Module):
    """Head for sentence-level classification tasks."""
    def __init__(self, config):
        super().__init__()
        self.dense = nn.Linear(config.hidden_size, config.hidden_size)
        self.dropout = nn.Dropout(config.hidden_dropout_prob)
        self.out_proj = nn.Linear(config.hidden_size, 2)

    def forward(self, features, **kwargs):
        x = features[:, 0, :]  # <s> token (CLS)
        x = self.dropout(x)
        x = self.dense(x)
        x = torch.tanh(x)
        x = self.dropout(x)
        x = self.out_proj(x)
        return x


class Model(nn.Module):
    def __init__(self, encoder, config, tokenizer, args):
        super().__init__()
        self.encoder = encoder
        self.tokenizer = tokenizer
        self.classifier = RobertaClassificationHead(config)
        self.args = args

    def forward(self, input_embed=None, labels=None, output_attentions=False, input_ids=None):
        if output_attentions:
            if input_ids is not None:
                outputs = self.encoder.roberta(
                    input_ids,
                    attention_mask=input_ids.ne(1),
                    output_attentions=output_attentions,
                )
            else:
                outputs = self.encoder.roberta(
                    inputs_embeds=input_embed,
                    output_attentions=output_attentions,
                )
            attentions = outputs.attentions
            last_hidden_state = outputs.last_hidden_state
            logits = self.classifier(last_hidden_state)
            prob = torch.softmax(logits, dim=-1)
            return prob, attentions
        else:
            if input_ids is not None:
                outputs = self.encoder.roberta(
                    input_ids,
                    attention_mask=input_ids.ne(1),
                    output_attentions=output_attentions,
                )[0]
            else:
                outputs = self.encoder.roberta(
                    inputs_embeds=input_embed,
                    output_attentions=output_attentions,
                )[0]
            logits = self.classifier(outputs)
            prob = torch.softmax(logits, dim=-1)
            return prob


# ── Download fine-tuned weights from Google Drive ────────────────────────────
WEIGHTS_PATH = "12heads_linevul_model.bin"
if not os.path.exists(WEIGHTS_PATH):
    print("Downloading LineVul weights (~500 MB)...")
    os.system(f"gdown 'https://drive.google.com/uc?id=1oodyQqRb9jEcvLMVVKILmu8qHyNwd-zH' -O {WEIGHTS_PATH}")
else:
    print(f"Weights already present: {WEIGHTS_PATH}")

# ── Load base architecture from HuggingFace, then overlay fine-tuned weights ─
BASE_MODEL = "microsoft/codebert-base"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading base model ({BASE_MODEL}) on {DEVICE}...")
config = RobertaConfig.from_pretrained(BASE_MODEL)
config.num_labels = 1
config.attn_implementation = "eager"
tokenizer = RobertaTokenizer.from_pretrained(BASE_MODEL)

encoder = RobertaForSequenceClassification.from_pretrained(
    BASE_MODEL, config=config, ignore_mismatched_sizes=True, attn_implementation="eager"
)
model = Model(encoder, config, tokenizer, argparse.Namespace())
model.load_state_dict(torch.load(WEIGHTS_PATH, map_location=DEVICE), strict=False)
model.to(DEVICE)
model.eval()
print(f"LineVul loaded. Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Device: {DEVICE}")

In [ ]:
# ── 4. Core scoring logic ─────────────────────────────────────────────────────
import threading
import numpy as np
from typing import Dict, Tuple

MAX_TOKENS = 512   # RoBERTa/CodeBERT max sequence length
SA = 1.5           # paper constant (formula 9)
CAP_PERCENTILE = 90  # top-10% capping (formulas 7-8)


def _score_single_block(code: str) -> float:
    """Compute raw attention score for one basic block (formula 4).

    w_orig(m) = sum_{layers} sum_{content_tokens} attn[CLS -> token]

    The CLS token aggregates global sequence context; its attention
    distribution highlights which tokens drive the vulnerability prediction.
    Blocks whose content tokens receive more CLS attention score higher.
    """
    encoded = tokenizer(
        code,
        max_length=MAX_TOKENS,
        truncation=True,
        padding="max_length",
        return_tensors="pt",
    )
    input_ids = encoded["input_ids"].to(DEVICE)
    # model.forward uses input_ids.ne(1) as the attention mask (pad id = 1)

    with torch.no_grad():
        prob, attentions = model(input_ids=input_ids, output_attentions=True)

    # attentions: tuple of 12 tensors, each [1, 12 heads, 512, 512]
    real_len = int(encoded["attention_mask"].sum().item())
    total_score = 0.0
    for layer_attn in attentions:
        # average over heads -> [512, 512]
        mean_attn = layer_attn[0].mean(dim=0)
        # CLS row (position 0): attention CLS pays to every other token
        cls_row = mean_attn[0]
        # content tokens: skip CLS (idx 0) and SEP (idx real_len-1)
        content_att = cls_row[1 : real_len - 1]
        total_score += float(content_att.sum().cpu())

    return total_score


def _normalize(raw: Dict[str, float]) -> Dict[str, float]:
    """Min-max normalize raw scores to [0, 0.5] with 90th-percentile cap."""
    if not raw:
        return {}
    values = np.array(list(raw.values()), dtype=float)
    w_max = float(np.percentile(values, CAP_PERCENTILE))
    w_min = float(values.min())
    normalized: Dict[str, float] = {}
    for bb_id, s in raw.items():
        if w_min == w_max:
            normalized[bb_id] = 0.5
        else:
            normalized[bb_id] = 0.5 * (min(s, w_max) - w_min) / (w_max - w_min)
    return normalized


def score_blocks(blocks: Dict[str, str]) -> Tuple[Dict[str, float], Dict[str, float]]:
    """Return (raw_scores, normalized_scores) for all blocks."""
    raw = {bb_id: _score_single_block(code) for bb_id, code in blocks.items()}
    return raw, _normalize(raw)


def compute_attention_distances(
    blocks: Dict[str, str],
    physical_distances: Dict[str, float],
) -> Tuple[Dict[str, float], Dict[str, float]]:
    """Formula 9: db_att(m) = db_phys(m) * (SA - w(m))."""
    raw, normalized = score_blocks(blocks)
    att_distances = {
        bb_id: phys * (SA - normalized.get(bb_id, 0.5))
        for bb_id, phys in physical_distances.items()
    }
    return att_distances, normalized


print("Scoring logic ready.")

In [ ]:
# ── 5. Define FastAPI app ─────────────────────────────────────────────────────
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

app = FastAPI(title="LineVul Attention Distance Server", version="1.0")

_metrics_lock = threading.Lock()
_metrics = {"blocks_scored": 0, "requests": 0}


# ── /score_blocks ─────────────────────────────────────────────────────────────
class ScoreRequest(BaseModel):
    blocks: Dict[str, str]  # bb_id -> source code

class ScoreResponse(BaseModel):
    raw_scores: Dict[str, float]        # w_orig before normalization
    normalized_scores: Dict[str, float]  # w(m) in [0, 0.5]


@app.post("/score_blocks", response_model=ScoreResponse)
def score_blocks_endpoint(req: ScoreRequest):
    try:
        raw, normalized = score_blocks(req.blocks)
        with _metrics_lock:
            _metrics["blocks_scored"] += len(raw)
            _metrics["requests"] += 1
        return ScoreResponse(raw_scores=raw, normalized_scores=normalized)
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


# ── /compute_distances ────────────────────────────────────────────────────────
class DistanceRequest(BaseModel):
    blocks: Dict[str, str]              # bb_id -> source code
    physical_distances: Dict[str, float]  # bb_id -> AFLGo physical distance

class DistanceResponse(BaseModel):
    attention_distances: Dict[str, float]  # db_att = db_phys * (1.5 - w(m))
    normalized_scores: Dict[str, float]    # w(m) for inspection


@app.post("/compute_distances", response_model=DistanceResponse)
def compute_distances_endpoint(req: DistanceRequest):
    try:
        att_distances, normalized = compute_attention_distances(
            req.blocks, req.physical_distances
        )
        with _metrics_lock:
            _metrics["blocks_scored"] += len(req.blocks)
            _metrics["requests"] += 1
        return DistanceResponse(attention_distances=att_distances, normalized_scores=normalized)
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


# ── /health & /metrics ────────────────────────────────────────────────────────
@app.get("/health")
def health():
    return {"status": "ok", "model": BASE_MODEL, "device": DEVICE}


@app.get("/metrics")
def metrics():
    with _metrics_lock:
        return dict(_metrics)


print("FastAPI app defined.")

In [ ]:
# ── 6. Start uvicorn in a background thread ───────────────────────────────────
import uvicorn

PORT = 8001  # avoids conflict if Qwen server (8000) runs in another session

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=PORT, log_level="info")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
print(f"Uvicorn started on port {PORT}")

In [ ]:
# ── 7. Expose with ngrok ──────────────────────────────────────────────────────
import time
time.sleep(2)  # give uvicorn a moment to bind

tunnel = ngrok.connect(PORT, "http")
public_url = tunnel.public_url

print("=" * 60)
print(f"  Public URL        : {public_url}")
print(f"  Health            : {public_url}/health")
print(f"  Docs              : {public_url}/docs")
print(f"  Score blocks      : POST {public_url}/score_blocks")
print(f"  Compute distances : POST {public_url}/compute_distances")
print("=" * 60)
print()
print("Paste the Public URL into configs/ as: attention_distance.server_url")

In [ ]:
# ── 8. Smoke test ─────────────────────────────────────────────────────────────
import requests, json

sample_blocks = {
    "main_bb0": "int main(int argc, char** argv) {\n  if (argc < 2) return 1;\n",
    "parse_bb1": "void parse_input(char* buf, size_t len) {\n  memcpy(dst, buf, len);\n",
    "safe_bb2":  "void log_message(const char* msg) {\n  printf(\"%s\", msg);\n",
}

resp = requests.post(
    f"{public_url}/score_blocks",
    json={"blocks": sample_blocks},
    timeout=120,
)
print("=== /score_blocks ===")
print(json.dumps(resp.json(), indent=2))

resp2 = requests.post(
    f"{public_url}/compute_distances",
    json={
        "blocks": sample_blocks,
        "physical_distances": {"main_bb0": 4.0, "parse_bb1": 1.5, "safe_bb2": 8.0},
    },
    timeout=120,
)
print("\n=== /compute_distances ===")
print(json.dumps(resp2.json(), indent=2))

## Client Usage

### Score basic blocks → get `w(m)` values
```python
import requests

SERVER_URL = "https://<your-ngrok-id>.ngrok-free.app"  # from cell 7

resp = requests.post(
    f"{SERVER_URL}/score_blocks",
    json={
        "blocks": {
            "func_A_bb0": "void parse(char* buf) {\n  if (!buf) return;\n",
            "func_A_bb1": "  memcpy(dst, buf, strlen(buf));\n}",
        }
    },
    timeout=120,
)
result = resp.json()
# result["normalized_scores"]["func_A_bb0"] -> w(m) in [0, 0.5]
```

### Compute attention distances directly
```python
resp = requests.post(
    f"{SERVER_URL}/compute_distances",
    json={
        "blocks": {
            "func_A_bb0": "void parse(char* buf) { ... }",
            "func_A_bb1": "  memcpy(dst, buf, len); }",
        },
        "physical_distances": {
            "func_A_bb0": 4.0,
            "func_A_bb1": 1.5,
        },
    },
    timeout=120,
)
result = resp.json()
# result["attention_distances"]["func_A_bb1"] -> db_att = 1.5 * (1.5 - w(m))
```

### Request / Response schema

**`POST /score_blocks`**
| Field | Type | Description |
|---|---|---|
| `blocks` | `{bb_id: code_str}` | Basic block source code, keyed by block ID |

Response: `{raw_scores: {bb_id: float}, normalized_scores: {bb_id: float}}`

**`POST /compute_distances`**
| Field | Type | Description |
|---|---|---|
| `blocks` | `{bb_id: code_str}` | Basic block source code |
| `physical_distances` | `{bb_id: float}` | AFLGo physical distance per block |

Response: `{attention_distances: {bb_id: float}, normalized_scores: {bb_id: float}}`

> **Colab GPU tip:** LineVul is ~125M params (CodeBERT-based). A single T4 (16 GB) is more than sufficient.  
> Throughput: ~50–200 blocks/second depending on block size.